In [ ]:
#Create a new notebook. Pick one stock — ABUK, HRHO, or any other from the universe. 
#Predict its daily return using an MLP. Split 70/30, train on the past, test on the future. 
#Plot: training loss vs. testing loss, and predicted returns vs. actual returns, for both the training and testing periods.

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path

from tradinglab.data_feed import DataFeed
from tradinglab.features import build_dataset, train_test_split, N_FEATURES
from tradinglab.models import MLP
from tradinglab.ml import train_model, predict

In [ ]:
# ---- 1. Discover the universe and load it ----
all_csvs = sorted(p.stem for p in Path('data/egx').glob('*.csv'))
print('files found:', all_csvs)

# egx30.csv is the INDEX, not a tradeable stock -- exclude it from the universe.
# It has a different schema (Date, Price, Open...) and is loaded separately,
# only as a benchmark, via data_feed.load_egx30_returns.
stock_symbols = [s for s in all_csvs if s.lower() != 'egx30']

feed = DataFeed.from_dir('data/egx', symbols=stock_symbols)
print('universe loaded:', feed.symbols)
print('date range:', feed.dates[0].date(), '->', feed.dates[-1].date(), f'({feed.n_days} days)')


In [ ]:
# ---- 2. Pick one stock ----
ticker = 'ABUK'                       # change to 'HRHO' or any symbol printed above
asset_idx = feed.symbols.index(ticker)

In [ ]:
# ---- 3. Build the dataset -- SINGLE LAG version, direct parallel to the sine wave ----
X_full, y = build_dataset(feed, asset_idx)   # X_full has 9 engineered features, in order
X = X_full[:, :1]                            # keep only column 0: "return" (today's return)

Xtr, ytr, Xte, yte = train_test_split(X, y)  # chronological 70/30 -- test IS the future

print(f'total samples: {len(X)}   train: {len(Xtr)}   test: {len(Xte)}')

In [ ]:
# ---- 4. Train -- same MLP shape as the lesson, done directly here ----
torch.manual_seed(0)
model = MLP(n_features=1, hidden=32)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# convert explicitly, bypassing torch.as_tensor's dtype inference
Xtr_t = torch.tensor(np.ascontiguousarray(Xtr, dtype=np.float32))
ytr_t = torch.tensor(np.ascontiguousarray(ytr, dtype=np.float32))
Xte_t = torch.tensor(np.ascontiguousarray(Xte, dtype=np.float32))
yte_t = torch.tensor(np.ascontiguousarray(yte, dtype=np.float32))

history = {'train': [], 'test': []}
for epoch in range(300):
    model.train()
    opt.zero_grad()
    loss = loss_fn(model(Xtr_t), ytr_t)
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        test_loss = loss_fn(model(Xte_t), yte_t).item()
    history['train'].append(loss.item())
    history['test'].append(test_loss)

pred_train = model(Xtr_t).detach().numpy()
pred_test = model(Xte_t).detach().numpy()

zero_baseline = float(np.mean(yte**2))
persistence_baseline = float(np.mean((Xte[:, 0] - yte)**2))

print(f'final train loss: {history["train"][-1]:.6f}')
print(f'final test loss:  {history["test"][-1]:.6f}')
print(f'"predict zero" baseline:      {zero_baseline:.6f}')
print(f'"predict no change" baseline: {persistence_baseline:.6f}')

In [ ]:
# ---- 5. Plots: loss curves, and predicted vs actual for BOTH periods ----
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].plot(history['train'], label='train loss')
axes[0].plot(history['test'], label='test loss')
axes[0].set_yscale('log'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[0].set_title(f'{ticker}: loss curves')

plt.tight_layout(); plt.show()

In [ ]:
# ---- 6. Accuracy Metrics 
from tradinglab.metrics import directional_accuracy, information_coefficient

# flatten predictions to 1-D to match yte's shape
pred_test_flat = pred_test.ravel()
pred_train_flat = pred_train.ravel()

da_test = directional_accuracy(pred_test_flat, yte)
da_train = directional_accuracy(pred_train_flat, ytr)
ic_test = information_coefficient(pred_test_flat, yte)
ic_train = information_coefficient(pred_train_flat, ytr)

print(f'Training period:')
print(f'  directional accuracy: {da_train:.3f}   (0.5 = coin flip)')
print(f'  information coefficient: {ic_train:.4f}')
print()
print(f'Testing period:')
print(f'  directional accuracy: {da_test:.3f}   (0.5 = coin flip)')
print(f'  information coefficient: {ic_test:.4f}')


# ---- MSE and MAE ----
mse_train = float(np.mean((pred_train_flat - ytr)**2))
mse_test = float(np.mean((pred_test_flat - yte)**2))

mae_train = float(np.mean(np.abs(pred_train_flat - ytr)))
mae_test = float(np.mean(np.abs(pred_test_flat - yte)))

print(f'Training period:')
print(f'  MSE: {mse_train:.6f}')
print(f'  MAE: {mae_train:.6f}')
print()
print(f'Testing period:')
print(f'  MSE: {mse_test:.6f}')
print(f'  MAE: {mae_test:.6f}')